In [13]:
!pip install accelerate



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Filename: csv_query_assistant.py

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import warnings
import io
import sys
from contextlib import redirect_stdout

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module='transformers.utils.import_utils')
warnings.filterwarnings("ignore", message="Torch AMP is not available") # Common on CPU

# --- Global Variables ---
LLM_MODEL = None
LLM_TOKENIZER = None
TEXT_GENERATOR_PIPELINE = None

# --- LLM and Pandas Interaction ---

def load_llm_model(model_id: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"): # Defaulting to TinyLlama
    """
    Loads the LLM model and tokenizer from Hugging Face.
    Uses 4-bit quantization if CUDA is available.
    """
    global LLM_MODEL, LLM_TOKENIZER, TEXT_GENERATOR_PIPELINE
    if TEXT_GENERATOR_PIPELINE is not None:
        print("LLM model already loaded.")
        return True

    print(f"Attempting to load LLM: {model_id}")
    try:
        model_kwargs = {"device_map": "auto"} # Common arguments

        if torch.cuda.is_available():
            print("CUDA available. Configuring 4-bit quantization.")
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
            model_kwargs["quantization_config"] = bnb_config
            model_kwargs["torch_dtype"] = torch.bfloat16 # For GPU
            
            LLM_TOKENIZER = AutoTokenizer.from_pretrained(model_id)
            LLM_MODEL = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
            print("Model loaded with 4-bit quantization on GPU.")
        else:
            print("CUDA not available. Loading model on CPU (this may be slow and RAM intensive).")
            # Do NOT pass quantization_config for CPU
            # device_map="auto" will handle CPU placement.
            # torch_dtype can be float32 for CPU for broader compatibility.
            model_kwargs["torch_dtype"] = torch.float32 # Safer default for CPU
            
            LLM_TOKENIZER = AutoTokenizer.from_pretrained(model_id)
            LLM_MODEL = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
            print(f"Model loaded on CPU with dtype: {LLM_MODEL.dtype}.")


        if LLM_TOKENIZER.pad_token_id is None:
            print("Setting pad_token_id to eos_token_id for tokenizer.")
            LLM_TOKENIZER.pad_token_id = LLM_TOKENIZER.eos_token_id
        
        TEXT_GENERATOR_PIPELINE = pipeline(
            "text-generation",
            model=LLM_MODEL,
            tokenizer=LLM_TOKENIZER,
            # The pipeline will infer the dtype from the model, but can be set explicitly
            # torch_dtype=LLM_MODEL.dtype, # Use the model's loaded dtype
            device=LLM_MODEL.device # Ensure pipeline uses the same device as the model
        )
        print(f"Text generation pipeline initialized on device: {TEXT_GENERATOR_PIPELINE.device}")
        return True
        
    except Exception as e:
        print(f"Error loading LLM model: {e}")
        print("Please ensure you have requested access for gated models (if applicable) and are logged in via `huggingface-cli login`.")
        print("Also ensure 'accelerate' is installed (`pip install accelerate`).")
        LLM_MODEL, LLM_TOKENIZER, TEXT_GENERATOR_PIPELINE = None, None, None
        return False

def get_dataframe_schema(df: pd.DataFrame) -> str:
    """Generates a string representation of the DataFrame's schema."""
    schema_parts = ["DataFrame `df` has the following columns and data types:"]
    for column, dtype in df.dtypes.items():
        schema_parts.append(f"- Column '{column}' (dtype: {dtype})")
    
    if not df.empty:
        schema_parts.append("\nHere's a small sample of the data (first 2 rows):")
        sample_df_string = df.head(2).to_string(index=False)
        schema_parts.append(sample_df_string)
    return "\n".join(schema_parts)

def generate_pandas_code(df_schema: str, natural_language_query: str) -> str:
    """
    Uses the LLM to generate Pandas code based on the schema and query.
    """
    if TEXT_GENERATOR_PIPELINE is None:
        return "Error: LLM pipeline not initialized."

    system_prompt = (
        "You are an expert Python Pandas code generation assistant. "
        "Given a natural language query about a Pandas DataFrame named `df` and its schema, "
        "you MUST generate ONLY the Python code (using the `df` DataFrame) that would answer the query. "
        "The code should typically result in a printable output (e.g., a DataFrame, Series, value, or use `print()`). "
        "Do not add any explanations, introductions, or markdown formatting like ```python ... ```. "
        "Just the raw Python code. Ensure the code is a single block."
        "If the query asks for a specific value, assign it to a variable named 'result' and then print 'result'. "
        "If the query asks to display data (like 'show rows' or 'display info'), use `print(df_subset)` or `print(df.info())` etc."
    )
    
    user_prompt_content = (
        f"{df_schema}\n\n"
        f"Natural language query: \"{natural_language_query}\"\n\n"
        "Generate the Python Pandas code to answer this query using the `df` DataFrame. "
        "The code must produce a printable output."
    )

    # For chat models like TinyLlama-Chat, we need to format the input as a conversation.
    # The pipeline usually handles this if the tokenizer has a chat_template.
    # If not, manual formatting is needed. TinyLlama's template is usually something like:
    # <|system|>
    # {system_prompt}</s>
    # <|user|>
    # {user_prompt_content}</s>
    # <|assistant|>
    #
    # The pipeline with a chat model should ideally handle this.
    # We'll construct the messages list for clarity and compatibility.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_content},
    ]

    print("\nAsking LLM to generate Pandas code...")
    try:
        outputs = TEXT_GENERATOR_PIPELINE(
            messages,
            max_new_tokens=200,  # Max tokens for the generated code
            do_sample=True,
            temperature=0.1,   # Lower temperature for more deterministic/factual code
            top_k=5,           # Consider a smaller top_k for code generation
            top_p=0.9,
            eos_token_id=LLM_TOKENIZER.eos_token_id,
            pad_token_id=LLM_TOKENIZER.pad_token_id  # Ensure pad_token_id is used
        )
        
        generated_text = ""
        # Output format for pipeline with messages can be a list of conversations
        if outputs and isinstance(outputs, list) and isinstance(outputs[0], list) and 'content' in outputs[0][-1]:
            # Assuming the last message in the list is the assistant's response
            if outputs[0][-1]['role'] == 'assistant':
                generated_text = outputs[0][-1]['content']
        elif outputs and isinstance(outputs, list) and 'generated_text' in outputs[0]: # Older pipeline format
             # This part handles if the output is a dict with 'generated_text'
             # which itself might be a full conversation string or just the assistant's part
            full_response = outputs[0]['generated_text']
            if isinstance(full_response, list): # If generated_text is a list of chat messages
                 for msg_part in full_response:
                    if msg_part['role'] == 'assistant':
                        generated_text += msg_part['content']
                        break # Assuming one assistant response part
            elif isinstance(full_response, str):
                # If it's a string, it might be the full conversation including prompts.
                # We need to extract just the assistant's response.
                # This is tricky without knowing the exact template used by the pipeline.
                # A common way for TinyLlama if not handled by pipeline:
                assistant_marker_str = "<|assistant|>" # Or similar based on model's actual template
                idx = full_response.rfind(assistant_marker_str)
                if idx != -1:
                    generated_text = full_response[idx + len(assistant_marker_str):].strip()
                else: # Fallback: assume the whole string is the response (might contain prompt)
                    generated_text = full_response 
        else:
            print(f"Unexpected LLM output format: {outputs}")


        code = generated_text.strip()
        # More robust cleaning for code blocks
        if "```python" in code:
            code = code.split("```python")[1].strip()
        if "```" in code:
            code = code.split("```")[0].strip()
        
        lines = code.split('\n')
        cleaned_lines = []
        for line in lines:
            # Remove lines that are just explanations or "Sure, here is..."
            if not (line.strip().lower().startswith("sure, here") or \
                    line.strip().lower().startswith("here is the python") or \
                    line.strip().lower().startswith("the python code is:") or \
                    line.strip().lower().startswith("here's the code")):
                cleaned_lines.append(line)
        code = "\n".join(cleaned_lines).strip()

        if not code:
            return "Error: LLM generated empty or non-code response."
        return code

    except Exception as e:
        print(f"Error during LLM code generation: {e}")
        return f"Error: Could not generate code - {e}"

def execute_generated_code(df: pd.DataFrame, generated_code: str):
    """
    Executes the LLM-generated Pandas code.
    WARNING: THIS IS RISKY. FOR DEMONSTRATION ONLY.
    """
    print("\n--- EXECUTING GENERATED CODE ---")
    print("--- SECURITY WARNING: Executing LLM-generated code can be dangerous. ---")
    print("--- Review the code carefully before execution in a real application. ---")
    print("Generated Code to Execute:")
    print(f"{'-'*30}\n{generated_code}\n{'-'*30}")

    # Simple check for potentially harmful keywords - VERY basic, not a real sandbox
    restricted_keywords = ['os.', 'sys.', 'subprocess.', 'eval(', 'exec(', '__import__', 'open(']
    for keyword in restricted_keywords:
        if keyword in generated_code:
            return f"Error: Generated code contains potentially unsafe keyword '{keyword}'. Execution aborted."

    local_scope = {'df': df.copy(), 'pd': pd, 'result': None} 
    
    stdout_capture = io.StringIO()
    try:
        with redirect_stdout(stdout_capture):
            # Using a restricted global scope for exec
            exec(generated_code, {"pd": pd, "__builtins__": {"print": print, "range": range, "len": len, "list": list, "dict": dict, "str": str, "int": int, "float": float, "True": True, "False": False, "None": None, "abs": abs, "round": round, "sum": sum, "min": min, "max": max, "sorted": sorted}}, local_scope) 
        
        execution_output = stdout_capture.getvalue()
        result_variable = local_scope.get('result', None) # Check if 'result' was assigned

        output_message = "--- Execution Result ---"
        if execution_output:
            output_message += "\nPrinted Output:\n" + execution_output.strip()
        
        if result_variable is not None:
             # Check if result_variable itself was printed. If not, print it.
            if not execution_output or str(result_variable) not in execution_output:
                 output_message += f"\nValue of 'result' variable (not explicitly printed by generated code):\n{result_variable}"
        elif not execution_output:
             output_message += "\n(No explicit output from generated code, e.g., no print() statement or 'result' variable assigned and printed)"
        
        return output_message.strip()

    except Exception as e:
        error_msg = f"Error executing generated code: {type(e).__name__}: {e}\n"
        if stdout_capture.getvalue():
            error_msg += f"Captured stdout before error:\n{stdout_capture.getvalue().strip()}"
        return error_msg.strip()


# --- Main Application Logic ---

def main():
    """Main function to run the CSV Query Assistant."""
    print("--- CSV Data Analysis Assistant (LLM + Pandas) ---")
    
    # Explicitly set the model_id here for clarity
    # model_to_use = "meta-llama/Meta-Llama-3-8B-Instruct" # If you have access
    model_to_use = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # For testing
    
    if not load_llm_model(model_id=model_to_use): 
        print("Failed to load LLM. Exiting application.")
        return

    csv_path = input("Enter the path to your CSV file (e.g., sample_data.csv): ").strip()
    try:
        df = pd.read_csv(csv_path)
        print(f"\nSuccessfully loaded '{csv_path}'.")
        print("DataFrame Info:")
        # Create a string buffer to capture df.info() output
        info_buffer = io.StringIO()
        df.info(buf=info_buffer)
        print(info_buffer.getvalue())
        print("\nFirst 5 rows of your data (df.head()):")
        print(df.head())
    except FileNotFoundError:
        print(f"Error: CSV file not found at '{csv_path}'. Please check the path and try again.")
        return
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    df_schema_str = get_dataframe_schema(df)

    while True:
        print("\n" + "="*40)
        natural_language_query = input("Ask a question about your data (or type 'exit' to quit): \n> ").strip()

        if natural_language_query.lower() == 'exit':
            print("Exiting CSV Query Assistant. Goodbye!")
            break
        
        if not natural_language_query:
            print("Please enter a query.")
            continue

        generated_code = generate_pandas_code(df_schema_str, natural_language_query)
        
        if generated_code.startswith("Error:"):
            print(f"\nLLM Code Generation Failed:\n{generated_code}")
        else:
            print(f"\nLLM Suggested Code:\n{'-'*30}\n{generated_code}\n{'-'*30}")
            confirm_exec = input("Execute this code? (yes/no) [yes]: ").strip().lower()
            if confirm_exec == "" or confirm_exec == "yes" or confirm_exec == "y":
                execution_result = execute_generated_code(df, generated_code)
                print(execution_result)
            else:
                print("Execution cancelled by user.")

if __name__ == "__main__":
    main()


--- CSV Data Analysis Assistant (LLM + Pandas) ---
Attempting to load LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0
CUDA not available. Loading model on CPU (this may be slow and RAM intensive).


model.safetensors:  12%|#2        | 304M/2.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Model loaded on CPU with dtype: torch.float32.
Error loading LLM model: The model has been loaded with `accelerate` and therefore cannot be moved to a specific device. Please discard the `device` argument when creating your pipeline object.
Please ensure you have requested access for gated models (if applicable) and are logged in via `huggingface-cli login`.
Also ensure 'accelerate' is installed (`pip install accelerate`).
Failed to load LLM. Exiting application.
